In [1]:
import sys
import os
import time
import gc

sys.path.append("..")

import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModel

from common import *

os.environ["TOKENIZERS_PARALLELISM"] = "false"

torch.set_num_threads(12)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Languages:", LANGS)

bench = {
    lang: load_benchmark(lang)
    for lang in LANGS
}

for lang in LANGS:
    q, c, gold = bench[lang]
    print(
        f"{lang}: queries={len(q)}, "
        f"docs={len(c)}, "
        f"gold={len(gold)}"
    )

Torch: 2.14.1+cpu
CUDA available: False
Languages: ['as', 'bn', 'gu', 'hi', 'kn', 'ml', 'mr', 'ne', 'or', 'pa', 'ta', 'te', 'ur']
as: queries=1000, docs=1000, gold=1000
bn: queries=1000, docs=1000, gold=1000
gu: queries=999, docs=999, gold=999
hi: queries=1000, docs=1000, gold=1000
kn: queries=1000, docs=1000, gold=1000
ml: queries=1000, docs=1000, gold=1000
mr: queries=1000, docs=1000, gold=1000
ne: queries=1000, docs=1000, gold=1000
or: queries=1000, docs=1000, gold=1000
pa: queries=1000, docs=1000, gold=1000
ta: queries=1000, docs=1000, gold=1000
te: queries=1000, docs=1000, gold=1000
ur: queries=1000, docs=1000, gold=1000


In [2]:
MURIL_ID = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(
    MURIL_ID
)

model = AutoModel.from_pretrained(
    MURIL_ID
)

model.eval()

print("Model:", MURIL_ID)
print("Hidden size:", model.config.hidden_size)
print("Max position embeddings:", model.config.max_position_embeddings)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

c:\Users\sanka\OneDrive\Desktop\research paper\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sanka\.cache\huggingface\hub\models--google--muril-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

...

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: google/muril-base-cased
Hidden size: 768
Max position embeddings: 512


In [3]:
def mean_pooling(
    last_hidden_state,
    attention_mask
):
    """
    Attention-mask-weighted mean pooling.

    last_hidden_state:
        [batch, seq_len, hidden_size]

    attention_mask:
        [batch, seq_len]
    """

    mask = attention_mask.unsqueeze(-1).to(
        last_hidden_state.dtype
    )

    masked_embeddings = (
        last_hidden_state * mask
    )

    summed = masked_embeddings.sum(dim=1)

    counts = mask.sum(dim=1).clamp(
        min=1e-9
    )

    return summed / counts


def l2_normalize(embeddings):
    """
    Row-wise L2 normalization.
    """

    return embeddings / (
        np.linalg.norm(
            embeddings,
            axis=1,
            keepdims=True
        ) + 1e-12
    )

In [4]:
def encode_hf_encoder(
    tokenizer,
    model,
    texts,
    cache_path,
    batch_size=16,
    max_length=256,
):
    """
    Encode texts using:
        tokenizer
        -> final hidden states
        -> attention-mask mean pooling
        -> L2 normalization

    Cached embeddings are stored as .npy.
    """

    cache_path = Path(cache_path)

    if cache_path.exists():
        print("Loading cached:", cache_path)
        return np.load(cache_path)

    model.eval()

    all_embeddings = []

    with torch.no_grad():

        for start in range(
            0,
            len(texts),
            batch_size
        ):

            batch_texts = texts[
                start:start + batch_size
            ]

            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )

            outputs = model(
                **encoded
            )

            pooled = mean_pooling(
                outputs.last_hidden_state,
                encoded["attention_mask"],
            )

            pooled = pooled.cpu().numpy()

            pooled = l2_normalize(
                pooled
            )

            all_embeddings.append(
                pooled.astype(np.float32)
            )

            print(
                f"\rEncoded "
                f"{min(start + batch_size, len(texts))}"
                f"/{len(texts)}",
                end=""
            )

    print()

    embeddings = np.vstack(
        all_embeddings
    ).astype(np.float32)

    np.save(
        cache_path,
        embeddings
    )

    print(
        "Saved:",
        cache_path,
        "| shape:",
        embeddings.shape
    )

    return embeddings

In [5]:
test_texts = [
    bench["hi"][0]["query"].iloc[0],
    bench["hi"][1]["text"].iloc[0],
]

test_embeddings = encode_hf_encoder(
    tokenizer=tokenizer,
    model=model,
    texts=test_texts,
    cache_path=EMB_DIR / "muril_test.npy",
    batch_size=2,
    max_length=256,
)

print("Embedding shape:", test_embeddings.shape)

print(
    "L2 norms:",
    np.linalg.norm(
        test_embeddings,
        axis=1
    )
)

Encoded 2/2
Saved: C:\Users\sanka\OneDrive\Desktop\research paper\embeddings\muril_test.npy | shape: (2, 768)
Embedding shape: (2, 768)
L2 norms: [1. 1.]


In [6]:
for lang in ["hi", "ta", "as"]:

    q, c, gold = bench[lang]

    lengths = [
        len(
            tokenizer.encode(
                text,
                add_special_tokens=True
            )
        )
        for text in c["text"]
    ]

    lengths = np.array(lengths)

    print(
        lang,
        "| median:",
        int(np.median(lengths)),
        "| 90%:",
        int(np.percentile(lengths, 90)),
        "| max:",
        int(lengths.max()),
        "| >256:",
        f"{(lengths > 256).mean():.1%}"
    )

model.safetensors: reconstructing file:   0%|          |  0.00B /  953MB            

model.safetensors: downloading bytes:           |  0.00B            

hi | median: 96 | 90%: 160 | max: 249 | >256: 0.0%


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (622 > 512). Running this sequence through the model will result in indexing errors


ta | median: 86 | 90%: 144 | max: 622 | >256: 0.1%
as | median: 92 | 90%: 154 | max: 254 | >256: 0.0%


In [7]:
lang = "hi"

q, c, gold = bench[lang]

t0 = time.time()

Eq = encode_hf_encoder(
    tokenizer,
    model,
    q["query"].tolist(),
    EMB_DIR / f"muril_{lang}_q.npy",
    batch_size=32,
    max_length=256,
)

Ed = encode_hf_encoder(
    tokenizer,
    model,
    c["text"].tolist(),
    EMB_DIR / f"muril_{lang}_d.npy",
    batch_size=16,
    max_length=256,
)

S = Eq @ Ed.T

print()
print("Query embeddings:", Eq.shape)
print("Document embeddings:", Ed.shape)
print("Score matrix:", S.shape)

metrics = record(
    "muril",
    lang,
    S,
    gold,
)

print(metrics)

print(
    f"Elapsed time: "
    f"{(time.time() - t0) / 60:.2f} min"
)

Encoded 1000/1000
Saved: C:\Users\sanka\OneDrive\Desktop\research paper\embeddings\muril_hi_q.npy | shape: (1000, 768)
Encoded 1000/1000
Saved: C:\Users\sanka\OneDrive\Desktop\research paper\embeddings\muril_hi_d.npy | shape: (1000, 768)

Query embeddings: (1000, 768)
Document embeddings: (1000, 768)
Score matrix: (1000, 1000)
{'MRR': 0.31423291254995295, 'MRR@10': 0.30164444444444444, 'R@1': 0.211, 'R@5': 0.422, 'R@10': 0.522, 'R@20': 0.611, 'R@100': 0.822, 'nDCG@10': 0.3539371478950248, 'n_queries': 1000, 'model': 'muril', 'key': 'hi', 'setting': 'mono', 'n_docs': 1000}
Elapsed time: 2.33 min


In [8]:
print(
    "Score min:",
    float(S.min())
)

print(
    "Score max:",
    float(S.max())
)

print(
    "Diagonal examples:",
    S[
        np.arange(min(5, len(S))),
        gold[:min(5, len(S))]
    ]
)

Score min: 0.9772032499313354
Score max: 0.9982413649559021
Diagonal examples: [0.99335074 0.99610627 0.994687   0.99491423 0.9920509 ]


In [9]:
results = []

for lang in LANGS:

    q, c, gold = bench[lang]

    print("\n" + "=" * 60)
    print("LANGUAGE:", lang)
    print("=" * 60)

    t0 = time.time()

    Eq = encode_hf_encoder(
        tokenizer,
        model,
        q["query"].tolist(),
        EMB_DIR / f"muril_{lang}_q.npy",
        batch_size=32,
        max_length=256,
    )

    Ed = encode_hf_encoder(
        tokenizer,
        model,
        c["text"].tolist(),
        EMB_DIR / f"muril_{lang}_d.npy",
        batch_size=16,
        max_length=256,
    )

    S = Eq @ Ed.T

    m = record(
        "muril",
        lang,
        S,
        gold,
    )

    results.append(m)

    print(
        f"{lang} | "
        f"MRR={m['MRR']:.4f} | "
        f"MRR@10={m['MRR@10']:.4f} | "
        f"R@1={m['R@1']:.4f} | "
        f"R@10={m['R@10']:.4f} | "
        f"R@100={m['R@100']:.4f} | "
        f"time={(time.time() - t0) / 60:.2f} min"
    )


LANGUAGE: as
Encoded 1000/1000
Saved: C:\Users\sanka\OneDrive\Desktop\research paper\embeddings\muril_as_q.npy | shape: (1000, 768)
Encoded 1000/1000
Saved: C:\Users\sanka\OneDrive\Desktop\research paper\embeddings\muril_as_d.npy | shape: (1000, 768)
as | MRR=0.2103 | MRR@10=0.1981 | R@1=0.1440 | R@10=0.3300 | R@100=0.6510 | time=2.36 min

LANGUAGE: bn
Encoded 1000/1000
Saved: C:\Users\sanka\OneDrive\Desktop\research paper\embeddings\muril_bn_q.npy | shape: (1000, 768)
Encoded 1000/1000
Saved: C:\Users\sanka\OneDrive\Desktop\research paper\embeddings\muril_bn_d.npy | shape: (1000, 768)
bn | MRR=0.3555 | MRR@10=0.3423 | R@1=0.2470 | R@10=0.5490 | R@100=0.8390 | time=1.98 min

LANGUAGE: gu
Encoded 999/999
Saved: C:\Users\sanka\OneDrive\Desktop\research paper\embeddings\muril_gu_q.npy | shape: (999, 768)
Encoded 999/999
Saved: C:\Users\sanka\OneDrive\Desktop\research paper\embeddings\muril_gu_d.npy | shape: (999, 768)
gu | MRR=0.3075 | MRR@10=0.2953 | R@1=0.2192 | R@10=0.4775 | R@100=0.7

In [10]:
metrics_df = pd.read_csv(
    RES_DIR / "metrics.csv"
)

muril_results = metrics_df[
    metrics_df["model"] == "muril"
][
    [
        "key",
        "MRR",
        "MRR@10",
        "R@1",
        "R@5",
        "R@10",
        "R@20",
        "R@100",
        "n_queries",
        "n_docs",
    ]
].sort_values("key")

print(
    muril_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

key    MRR  MRR@10    R@1    R@5   R@10   R@20  R@100  n_queries  n_docs
 as 0.2103  0.1981 0.1440 0.2640 0.3300 0.4060 0.6510       1000    1000
 bn 0.3555  0.3423 0.2470 0.4730 0.5490 0.6540 0.8390       1000    1000
 gu 0.3075  0.2953 0.2192 0.4014 0.4775 0.5656 0.7768        999     999
 hi 0.3142  0.3016 0.2110 0.4220 0.5220 0.6110 0.8220       1000    1000
 kn 0.4100  0.3995 0.3100 0.5120 0.6080 0.6920 0.8490       1000    1000
 ml 0.3688  0.3593 0.2700 0.4820 0.5650 0.6280 0.8110       1000    1000
 mr 0.3826  0.3724 0.2830 0.4990 0.5760 0.6520 0.8220       1000    1000
 ne 0.2782  0.2663 0.1980 0.3570 0.4520 0.5270 0.7490       1000    1000
 or 0.2546  0.2438 0.1800 0.3240 0.4040 0.4680 0.6790       1000    1000
 pa 0.3490  0.3373 0.2490 0.4530 0.5390 0.6290 0.8290       1000    1000
 ta 0.4074  0.3970 0.3060 0.5210 0.5930 0.6680 0.8300       1000    1000
 te 0.3275  0.3156 0.2400 0.4190 0.4920 0.5770 0.7720       1000    1000
 ur 0.2510  0.2384 0.1660 0.3330 0.4240 0.5130 0.73

In [11]:
print(
    "MuRIL mean MRR:",
    f"{muril_results['MRR'].mean():.4f}"
)

MuRIL mean MRR: 0.3244


In [12]:
del model
del tokenizer

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("MuRIL model released.")

MuRIL model released.


In [13]:
print("Query shape:", Eq.shape)
print("Document shape:", Ed.shape)

print(
    "Query norm mean:",
    np.linalg.norm(Eq, axis=1).mean()
)

print(
    "Document norm mean:",
    np.linalg.norm(Ed, axis=1).mean()
)

Query shape: (1000, 768)
Document shape: (1000, 768)
Query norm mean: 1.0
Document norm mean: 1.0


In [14]:
q, c, gold = bench["hi"]

S = (
    np.load(EMB_DIR / "muril_hi_q.npy")
    @
    np.load(EMB_DIR / "muril_hi_d.npy").T
)

real_mrr = metrics_from_ranks(
    gold_ranks(S, gold)
)["MRR"]

shuffled_gold = np.random.default_rng(1).permutation(gold)

shuffled_mrr = metrics_from_ranks(
    gold_ranks(S, shuffled_gold)
)["MRR"]

print("Real MuRIL MRR     :", round(real_mrr, 4))
print("Shuffled-gold MRR  :", round(shuffled_mrr, 4))

Real MuRIL MRR     : 0.3142
Shuffled-gold MRR  : 0.0081
